# POWER ANALYSIS

Statistical power analysis is a method used to determine the minimum sample size needed to detect an effect of a given size with a desired level of confidence.

This notebook covers:

* Effect Size Calculation (Cohen's d)
* Required Sample Size Estimation
* Power Curve Visualization
* Interpretation of Results


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.stats.power import TTestIndPower

In [ ]:
df = pd.read_csv('../data/processed/cleaned_marketing.csv')
df['date'] = pd.to_datetime(df['date'])

In [ ]:
control = df[df['group'] == 'control'].dropna(subset=['purchase'])
test = df[df['group'] == 'test'].dropna(subset=['purchase'])

print('Control group size:', len(control))
print('Test group size:', len(test))

## 1. Effect Size (Cohen's d)

Cohen's d measures the standardized difference between two means.
- Small effect: d = 0.2
- Medium effect: d = 0.5
- Large effect: d = 0.8


In [ ]:
mean_control = control['purchase'].mean()
mean_test = test['purchase'].mean()
std_control = control['purchase'].std()
std_test = test['purchase'].std()
n_control = len(control)
n_test = len(test)

# Pooled standard deviation
pooled_std = np.sqrt(
    ((n_control - 1) * std_control**2 + (n_test - 1) * std_test**2)
    / (n_control + n_test - 2)
)

cohens_d = (mean_control - mean_test) / pooled_std

print(f'Mean Purchase (Control): {mean_control:.2f}')
print(f'Mean Purchase (Test):    {mean_test:.2f}')
print(f'Pooled Std Dev:          {pooled_std:.2f}')
print(f"Cohen's d:               {cohens_d:.4f}")
print(f"Effect size magnitude:   {'Small' if abs(cohens_d) < 0.5 else 'Medium' if abs(cohens_d) < 0.8 else 'Large'}")

## 2. Required Sample Size

Given:
- Significance level (alpha) = 0.05
- Desired power (1 - beta) = 0.80
- Effect size = Cohen's d from above


In [ ]:
power_analysis = TTestIndPower()

alpha = 0.05
power = 0.80

# Use absolute value of Cohen's d
effect_size = abs(cohens_d)

# Calculate required sample size per group
required_n = power_analysis.solve_power(
    effect_size=effect_size,
    alpha=alpha,
    power=power,
    alternative='two-sided'
)

print(f'Required sample size per group: {required_n:.0f}')
print(f'Current control group size: {n_control}')
print(f'Current test group size:    {n_test}')
print()
if n_control >= required_n and n_test >= required_n:
    print('✓ Current sample sizes are sufficient.')
else:
    print('✗ Current sample sizes are INSUFFICIENT for desired power.')

## 3. Achieved Power

Given the current sample sizes, what is the actual statistical power of our test?


In [ ]:
achieved_power = power_analysis.solve_power(
    effect_size=effect_size,
    alpha=alpha,
    nobs1=n_control,
    alternative='two-sided'
)

print(f'Achieved Statistical Power: {achieved_power:.4f} ({achieved_power*100:.2f}%)')
print(f'Type II Error (Beta):       {1 - achieved_power:.4f} ({(1-achieved_power)*100:.2f}%)')

## 4. Power Curve

Visualizing how power changes as sample size increases for different effect sizes.


In [ ]:
sample_sizes = np.arange(5, 300, 5)
effect_sizes = [0.2, 0.5, 0.8, abs(cohens_d)]
labels = ['Small (d=0.2)', 'Medium (d=0.5)', 'Large (d=0.8)', f'Observed (d={abs(cohens_d):.3f})']

plt.figure(figsize=(10, 6))

for es, label in zip(effect_sizes, labels):
    powers = [
        power_analysis.solve_power(
            effect_size=es,
            alpha=0.05,
            nobs1=n,
            alternative='two-sided'
        )
        for n in sample_sizes
    ]
    plt.plot(sample_sizes, powers, label=label)

plt.axhline(y=0.80, color='red', linestyle='--', label='Target Power = 0.80')
plt.axvline(x=n_control, color='gray', linestyle=':', label=f'Current n = {n_control}')

plt.xlabel('Sample Size per Group')
plt.ylabel('Statistical Power')
plt.title('Power Curve by Effect Size')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Sample Size vs Alpha

How does the significance threshold affect required sample size?


In [ ]:
alphas = [0.01, 0.05, 0.10]
effect_size_vals = np.arange(0.1, 1.1, 0.1)

plt.figure(figsize=(10, 6))

for a in alphas:
    required_sizes = [
        power_analysis.solve_power(
            effect_size=es,
            alpha=a,
            power=0.80,
            alternative='two-sided'
        )
        for es in effect_size_vals
    ]
    plt.plot(effect_size_vals, required_sizes, label=f'alpha = {a}')

plt.xlabel("Effect Size (Cohen's d)")
plt.ylabel('Required Sample Size per Group')
plt.title('Required Sample Size vs Effect Size (Power = 0.80)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Summary Table


In [ ]:
summary = []
for a in [0.01, 0.05, 0.10]:
    for p in [0.70, 0.80, 0.90]:
        n = power_analysis.solve_power(
            effect_size=effect_size,
            alpha=a,
            power=p,
            alternative='two-sided'
        )
        summary.append({
            'Alpha': a,
            'Power': p,
            'Required N per group': int(np.ceil(n))
        })

summary_df = pd.DataFrame(summary)
print(f"Effect Size (Cohen's d) = {effect_size:.4f}")
print()
print(summary_df.to_string(index=False))

## INTERPRETATION

**Effect Size:** The observed Cohen's d is very small (close to 0), indicating that there is minimal practical difference in purchase counts between the Control and Test groups.

**Required Sample Size:** Because the effect size is so small, a very large sample size would be needed to reliably detect this difference with 80% power at the 5% significance level.

**Achieved Power:** With the current sample sizes (~29-30 per group), the achieved power is very low, meaning we have a high probability of failing to detect an effect even if one exists (Type II error).

**Recommendation:** The A/B test may be underpowered for the observed effect size. If the business expects a meaningful difference, either:
1. Run the experiment longer to collect more data, or
2. Re-evaluate whether the effect size is practically meaningful
